# AMANAH — Model Training and Validation Pipeline

This notebook runs the measured training, validation, packaging, and deployment pipeline for the AMANAH semantic-integrity model.

The pipeline keeps canonical Arabic and trusted reference translations immutable, prevents ayah-level leakage across dataset splits, calibrates thresholds on validation data only, evaluates the frozen test split, packages the model, and validates the inference contract before deployment.


In [ ]:
# 1. Resolve the repository from the notebook URL and initialize the workspace
from pathlib import Path
import os, re, json, shutil, subprocess, sys, platform

from google.colab import output

page_url = output.eval_js("window.location.href")
match = re.search(r"/github/([^/]+)/([^/]+)/blob/", page_url)
if not match:
    raise RuntimeError("Open this notebook from its GitHub-to-Colab link so the project repository can be resolved automatically.")

owner, repository = match.group(1), match.group(2)
repo_url = f"https://github.com/{owner}/{repository}.git"

workspace = Path("/content/amanah_project")
if workspace.exists():
    shutil.rmtree(workspace)

subprocess.run(["git","clone","--depth","1",repo_url,str(workspace)],check=True)
os.chdir(workspace)

print("Workspace:", workspace)
print("Python:", platform.python_version())


In [ ]:
# 2. Install dependencies and run deterministic quality gates
subprocess.run([sys.executable,"-m","pip","install","-q","-r","requirements.txt"],check=True)
subprocess.run([sys.executable,"-m","pytest","-q"],check=True)
subprocess.run([sys.executable,"-m","training.train","--smoke"],check=True)
subprocess.run([
    sys.executable,"-m","compileall","-q",
    "amanah_engine","api","data","training","scripts","deployment"
],check=True)

import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), "A GPU runtime is required for the measured fine-tuning stage."
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
# 3. Verify trusted-source contracts and retrieve the source bundle
subprocess.run([sys.executable,"scripts/source_contract_smoke.py","--timeout","90"],check=True)

Path("data/private").mkdir(parents=True,exist_ok=True)
Path("artifacts/amanah_sd").mkdir(parents=True,exist_ok=True)
Path("checkpoints").mkdir(parents=True,exist_ok=True)

subprocess.run([
    sys.executable,"scripts/fetch_verified_sources.py",
    "--output","data/private/reference_bundle.json",
    "--runtime-store","data/reference_store.json"
],check=True)

bundle=json.load(open("data/private/reference_bundle.json",encoding="utf-8"))
assert len(bundle["ayahs"])==6236
assert all(len(row["references"])==3 for row in bundle["ayahs"])
print("Verified canonical ayat:",len(bundle["ayahs"]))


In [ ]:
# 4. Build AMANAH-SD and enforce dataset QA
subprocess.run([
    sys.executable,"-m","scripts.prepare_amanah_sd",
    "data/private/reference_bundle.json",
    "artifacts/amanah_sd",
    "--seed","42"
],check=True)

subprocess.run([
    sys.executable,"scripts/qa_dataset.py",
    "--reference-bundle","data/private/reference_bundle.json",
    "--split-dir","artifacts/amanah_sd",
    "--output","artifacts/amanah_sd/qa_report.json"
],check=True)

qa=json.load(open("artifacts/amanah_sd/qa_report.json",encoding="utf-8"))
assert qa["status"]=="ok"
assert qa["split_ayah_leakage"]==[]
print(json.dumps(qa,ensure_ascii=False,indent=2))


In [ ]:
# 5. Fine-tune the semantic-drift classifier
subprocess.run([
    sys.executable,"-m","training.train",
    "--train","artifacts/amanah_sd/train.jsonl",
    "--validation","artifacts/amanah_sd/validation.jsonl",
    "--output-dir","checkpoints/amanah-drift-v0.1",
    "--epochs","5",
    "--batch-size","4",
    "--gradient-accumulation","4",
    "--learning-rate","2e-5",
    "--max-length","256",
    "--patience","2"
],check=True)


In [ ]:
# 6. Calibrate thresholds and evaluate the frozen held-out test split
subprocess.run([
    sys.executable,"-m","training.calibrate_thresholds",
    "--checkpoint","checkpoints/amanah-drift-v0.1",
    "--validation","artifacts/amanah_sd/validation.jsonl"
],check=True)

subprocess.run([
    sys.executable,"-m","training.evaluate",
    "--checkpoint","checkpoints/amanah-drift-v0.1",
    "--test","artifacts/amanah_sd/test.jsonl",
    "--output","artifacts/evaluation"
],check=True)

metrics=json.load(open("artifacts/evaluation/metrics.json",encoding="utf-8"))
assert metrics.get("status")=="ok"
for key in ["macro_f1","critical_drift_recall","false_safe_rate","severity_macro_f1"]:
    assert 0.0<=metrics[key]<=1.0
print(json.dumps(metrics,indent=2))


In [ ]:
# 7. Package the model and validate the inference handler locally
subprocess.run([
    sys.executable,"scripts/package_hf_model.py",
    "--checkpoint","checkpoints/amanah-drift-v0.1",
    "--reference-store","data/reference_store.json",
    "--output","artifacts/model_repository"
],check=True)

required=[
    "handler.py","model.pt","amanah_config.json",
    "thresholds.json","reference_store.json","requirements.txt"
]
missing=[name for name in required if not (Path("artifacts/model_repository")/name).exists()]
assert not missing,f"Missing deployment files: {missing}"

sys.path.insert(0,str(Path("artifacts/model_repository").resolve()))
from handler import EndpointHandler
handler=EndpointHandler("artifacts/model_repository")
store=json.load(open("artifacts/model_repository/reference_store.json",encoding="utf-8"))
row=store["ayahs"][0]

response=handler({"inputs":{
    "source_type":"quran",
    "source_ar":row["source_ar"],
    "candidate_en":row["references_en"][0],
    "ayah_id":row["ayah_id"]
}})
assert response["decision"] in {"PASS","REVIEW","CRITICAL","ABSTAIN"}
assert response["reference_status"]=="verified"
print(json.dumps(response,ensure_ascii=False,indent=2))


In [ ]:
# 8. Build the measured release report
report={
    "dataset_summary":json.load(open("artifacts/amanah_sd/dataset_summary.json",encoding="utf-8")),
    "dataset_qa":qa,
    "evaluation":metrics,
    "model_repository_url":None,
    "inference_endpoint_url":None
}

Path("artifacts/AMANAH_VALIDATION_REPORT.json").write_text(
    json.dumps(report,ensure_ascii=False,indent=2),
    encoding="utf-8"
)
shutil.make_archive(
    "/content/AMANAH_MODEL_RELEASE",
    "zip",
    "artifacts/model_repository"
)

print(json.dumps(report,ensure_ascii=False,indent=2))


In [ ]:
# 9. Publish the validated model package and deploy an authenticated endpoint
from google.colab import userdata
from huggingface_hub import (
    HfApi,
    create_repo,
    create_inference_endpoint,
    get_inference_endpoint,
    list_inference_endpoints_hardware,
    get_token,
    notebook_login
)

try:
    hf_token=userdata.get("HF_TOKEN")
except Exception:
    hf_token=None

if not hf_token:
    notebook_login()
    hf_token=get_token()

if not hf_token:
    raise RuntimeError("Hugging Face authentication was not completed.")

api=HfApi(token=hf_token)
identity=api.whoami()
namespace=identity.get("name") or identity.get("fullname")
model_repo_id=f"{namespace}/amanah-semantic-integrity-v0-1"

create_repo(
    repo_id=model_repo_id,
    repo_type="model",
    private=True,
    exist_ok=True,
    token=hf_token
)

api.upload_folder(
    repo_id=model_repo_id,
    repo_type="model",
    folder_path="artifacts/model_repository",
    token=hf_token,
    commit_message="Publish validated v0.1 checkpoint"
)

api.upload_file(
    path_or_fileobj="artifacts/AMANAH_VALIDATION_REPORT.json",
    path_in_repo="AMANAH_VALIDATION_REPORT.json",
    repo_id=model_repo_id,
    repo_type="model",
    token=hf_token,
    commit_message="Add measured validation report"
)

available=[
    hw for hw in list_inference_endpoints_hardware(namespace=namespace,token=hf_token)
    if str(getattr(hw,"status","")).lower()=="available"
]
cpu=[hw for hw in available if str(getattr(hw,"accelerator","")).lower()=="cpu"]
pool=cpu or available
if not pool:
    raise RuntimeError("No deployable Hugging Face Inference Endpoint hardware is currently available for this account.")

hardware=sorted(pool,key=lambda hw:float(getattr(hw,"price_per_hour",999999)))[0]

endpoint_name="amanah-semantic-integrity-v0-1"

try:
    endpoint=get_inference_endpoint(endpoint_name,namespace=namespace,token=hf_token)
    endpoint.update(
        repository=model_repo_id,
        framework="pytorch",
        task="custom",
        accelerator=hardware.accelerator,
        instance_size=hardware.instance_size,
        instance_type=hardware.instance_type,
        min_replica=0,
        max_replica=1,
        scale_to_zero_timeout=15
    )
except Exception:
    endpoint=create_inference_endpoint(
        name=endpoint_name,
        repository=model_repo_id,
        framework="pytorch",
        task="custom",
        accelerator=hardware.accelerator,
        vendor=hardware.vendor,
        region=hardware.region,
        instance_size=hardware.instance_size,
        instance_type=hardware.instance_type,
        min_replica=0,
        max_replica=1,
        scale_to_zero_timeout=15,
        type="authenticated",
        namespace=namespace,
        token=hf_token
    )

endpoint.wait(timeout=1200,refresh_every=10)
report["model_repository_url"]=f"https://huggingface.co/{model_repo_id}"
report["inference_endpoint_url"]=endpoint.url

Path("artifacts/AMANAH_VALIDATION_REPORT.json").write_text(
    json.dumps(report,ensure_ascii=False,indent=2),
    encoding="utf-8"
)

print("Model repository:",report["model_repository_url"])
print("Inference endpoint:",report["inference_endpoint_url"])


In [ ]:
# 10. Live endpoint smoke test
import requests

payload={"inputs":{
    "source_type":"quran",
    "source_ar":row["source_ar"],
    "candidate_en":row["references_en"][0],
    "ayah_id":row["ayah_id"]
}}

live=requests.post(
    report["inference_endpoint_url"],
    headers={
        "Authorization":f"Bearer {hf_token}",
        "Content-Type":"application/json"
    },
    json=payload,
    timeout=180
)
live.raise_for_status()
live_result=live.json()

assert live_result["decision"] in {"PASS","REVIEW","CRITICAL","ABSTAIN"}
assert live_result["reference_status"]=="verified"

print(json.dumps(live_result,ensure_ascii=False,indent=2))
print(json.dumps(report,ensure_ascii=False,indent=2))
